### CELL 1: Install All Dependencies

In [4]:
%pip install -q langchain langchain-community langchain-anthropic anthropic
%pip install -q langchain-core langchain-text-splitters
%pip install -q sentence-transformers faiss-cpu rank_bm25
%pip install -q ragas datasets
%pip install -q requests beautifulsoup4 pandas numpy matplotlib scikit-learn tqdm
print("All packages installed.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
All packages installed.


### CELL 2: Imports 

In [5]:
import os, re, time, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
from functools import wraps
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm
warnings.filterwarnings("ignore")

from langchain_anthropic import ChatAnthropic
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import faiss

from datasets import Dataset as HFDataset

print("✅ Imports successful.")

✅ Imports successful.


### CELL 3 — Config

In [9]:
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY not found in .env file")

COMPANIES = {
    "AAPL": "320193",
    "MSFT": "789019",
    "NVDA": "1045810",
    "JPM":  "19617",
    "XOM":  "34088",
}

CHUNK_SIZE_BASELINE = 512
CHUNK_SIZE_ENHANCED = 800
CHUNK_OVERLAP = 100
TOP_K = 5
HYBRID_K = 10
RRF_K = 60
DEDUP_THRESHOLD = 0.95

print(".env loaded and config ready.")

.env loaded and config ready.


### CELL 4: Utilities 

In [7]:

def with_retry(max_retries=3, base_delay=2.0):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries - 1:
                        raise
                    wait = base_delay * (2 ** attempt)
                    print(f"⚠ Retry {attempt+1}/{max_retries} after error: {e}")
                    time.sleep(wait)
        return wrapper
    return decorator

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, *args):
        self.elapsed_ms = (time.perf_counter() - self.start) * 1000

print("✅ Utilities ready.")

✅ Utilities ready.


### CELL 5: SEC EDGAR Fetching 

In [11]:

@with_retry(max_retries=3)
def fetch_url_text(url: str, headers: Dict) -> str:
    r = requests.get(url, headers=headers, timeout=25)
    r.raise_for_status()
    return r.text

def choose_primary_filing_doc(index_html: str) -> str:
    soup = BeautifulSoup(index_html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(" ", strip=True).lower()
        href_l = href.lower()
        if href_l.endswith((".htm", ".html")):
            links.append((href, text))

    preferred = []
    for href, text in links:
        score = 0
        h = href.lower()
        if "10-k" in h or "10k" in h:
            score += 5
        if "annual" in h:
            score += 3
        if "exhibit" in h:
            score -= 5
        if "index" in h:
            score -= 3
        preferred.append((score, href))

    preferred = sorted(preferred, reverse=True)
    return preferred[0][1] if preferred else None

def fetch_10k(cik: str, ticker: str) -> Tuple[str, str]:
    headers = {"User-Agent": "StudentAssignment student@warwick.ac.uk"}
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    meta = requests.get(submissions_url, headers=headers, timeout=15).json()
    recent = meta["filings"]["recent"]

    for i, form in enumerate(recent["form"]):
        if form != "10-K":
            continue

        acc = recent["accessionNumber"][i]
        acc_flat = acc.replace("-", "")
        filing_date = recent["filingDate"][i]

        index_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_flat}/{acc}-index.htm"
        index_html = fetch_url_text(index_url, headers)
        doc_path = choose_primary_filing_doc(index_html)

        if not doc_path:
            continue

        if doc_path.startswith("/"):
            filing_url = f"https://www.sec.gov{doc_path}"
        else:
            filing_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_flat}/{doc_path}"

        filing_html = fetch_url_text(filing_url, headers)
        text = BeautifulSoup(filing_html, "html.parser").get_text(separator=" ", strip=True)
        text = re.sub(r"\s{2,}", " ", text)

        print(f"✓ {ticker} | {filing_date} | {len(text):,} chars")
        return text, filing_date

    return "", ""

raw_docs = {}
for ticker, cik in COMPANIES.items():
    try:
        text, date = fetch_10k(cik, ticker)
        if text:
            raw_docs[ticker] = {"text": text, "date": date, "cik": cik}
        time.sleep(0.5)
    except Exception as e:
        print(f"✗ {ticker}: {e}")

print(f"Retrieved {len(raw_docs)}/{len(COMPANIES)} filings.")

✓ AAPL | 2025-10-31 | 73 chars
✓ MSFT | 2025-07-30 | 73 chars
✓ NVDA | 2026-02-25 | 73 chars
✓ JPM | 2026-02-13 | 73 chars
✓ XOM | 2026-02-18 | 73 chars
Retrieved 5/5 filings.


### CELL 6: Chunking

In [12]:

def baseline_chunk(docs: Dict) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_BASELINE,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = []
    for ticker, d in docs.items():
        for i, c in enumerate(splitter.split_text(d["text"])):
            if len(c.strip()) > 60:
                chunks.append(Document(
                    page_content=c,
                    metadata={"ticker": ticker, "chunk_id": i, "date": d["date"],
                              "section": "unstructured", "pipeline": "baseline"}
                ))
    print(f"✅ Baseline chunks: {len(chunks)}")
    return chunks

SEC_SECTIONS = {
    "Risk Factors": r'Item\s+1A[\.\s]*Risk\s+Factors',
    "Business": r'Item\s+1[\.\s]*Business',
    "MDA": r'Item\s+7[\.\s]*Management.{0,40}Discussion',
    "Quantitative": r'Item\s+7A[\.\s]*Quantitative',
    "Financials": r'Item\s+8[\.\s]*Financial\s+Statements',
    "Controls": r'Item\s+9A[\.\s]*Controls',
}

def enhanced_chunk(docs: Dict) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_ENHANCED,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = []
    for ticker, d in docs.items():
        text = d["text"]
        boundaries = sorted(
            [(m.start(), sec_name)
             for sec_name, pat in SEC_SECTIONS.items()
             for m in re.finditer(pat, text, re.IGNORECASE)],
            key=lambda x: x[0]
        )

        sections = []
        if not boundaries:
            sections = [("Full Document", text)]
        else:
            for i, (start, name) in enumerate(boundaries):
                end = boundaries[i+1][0] if i+1 < len(boundaries) else len(text)
                sections.append((name, text[start:end]))

        for sec_name, sec_text in sections:
            for i, c in enumerate(splitter.split_text(sec_text)):
                if len(c.strip()) > 80:
                    chunks.append(Document(
                        page_content=c,
                        metadata={"ticker": ticker, "chunk_id": i, "date": d["date"],
                                  "section": sec_name, "pipeline": "enhanced"}
                    ))
    print(f"✅ Enhanced chunks (pre-dedup): {len(chunks)}")
    return chunks

### CELL 7: Models + Dedup

In [15]:

embedder = SentenceTransformer("all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def deduplicate(chunks: List[Document], threshold: float = DEDUP_THRESHOLD) -> List[Document]:
    if not chunks:
        return chunks
    embs = embedder.encode([c.page_content for c in chunks], normalize_embeddings=True, show_progress_bar=False)
    keep = [0]
    for i in range(1, len(chunks)):
        sims = cosine_similarity([embs[i]], embs[keep])[0]
        if max(sims) < threshold:
            keep.append(i)
    deduped = [chunks[i] for i in keep]
    print(f"✅ Deduplicated {len(chunks)} → {len(deduped)}")
    return deduped

baseline_chunks = baseline_chunk(raw_docs)
enhanced_chunks = deduplicate(enhanced_chunk(raw_docs))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4909.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4574.27it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Baseline chunks: 5
✅ Enhanced chunks (pre-dedup): 0


In [16]:
print("baseline_chunks:", len(baseline_chunks))
print("enhanced_chunks:", len(enhanced_chunks))

if len(enhanced_chunks) > 0:
    print("Sample enhanced chunk:")
    print(enhanced_chunks[0].page_content[:300])
else:
    print("enhanced_chunks is empty")

baseline_chunks: 5
enhanced_chunks: 0
enhanced_chunks is empty


In [18]:
raw_enhanced = enhanced_chunk(raw_docs)
print("raw_enhanced:", len(raw_enhanced))

✅ Enhanced chunks (pre-dedup): 0
raw_enhanced: 0


In [20]:
def deduplicate(chunks: List[Document], threshold: float = 0.95) -> List[Document]:
    texts = [c.page_content.strip() for c in chunks if c.page_content and c.page_content.strip()]
    if len(texts) == 0:
        return []

    embs = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    embs = np.array(embs)

    keep = [0]
    for i in range(1, len(texts)):
        sims = cosine_similarity([embs[i]], embs[keep])[0]
        if max(sims) < threshold:
            keep.append(i)

    deduped = [chunks[i] for i in keep]
    print(f"✅ Deduplicated {len(chunks)} → {len(deduped)}")
    return deduped

In [22]:
enhanced_chunks = deduplicate(raw_enhanced)
print("enhanced_chunks after dedup:", len(enhanced_chunks))

enhanced_chunks after dedup: 0


### CELL 8: Indexing 

In [21]:

def build_faiss_index(chunks: List[Document]):
    texts = [c.page_content.strip() for c in chunks if c.page_content and c.page_content.strip()]

    if len(texts) == 0:
        raise ValueError("No valid chunk texts found. Check enhanced_chunk() or deduplicate().")

    embs = embedder.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    embs = np.array(embs, dtype="float32")

    if embs.ndim != 2:
        raise ValueError(f"Embeddings have wrong shape: {embs.shape}. Expected 2D array.")

    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index


def build_bm25(chunks: List[Document]):
    texts = [c.page_content.lower().split() for c in chunks if c.page_content and c.page_content.strip()]

    if len(texts) == 0:
        raise ValueError("No valid chunk texts for BM25.")

    return BM25Okapi(texts)


print("Building baseline FAISS...")
b_faiss = build_faiss_index(baseline_chunks)

print("Building enhanced FAISS...")
e_faiss = build_faiss_index(enhanced_chunks)

print("Building enhanced BM25...")
e_bm25 = build_bm25(enhanced_chunks)

print("✅ Indices ready.")

Building baseline FAISS...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.98it/s]

Building enhanced FAISS...


ValueError: No valid chunk texts found. Check enhanced_chunk() or deduplicate().